# Predikcija izpadov omrežja - Hackathon 2026
AB - Team

## Pridobivanje in označevanje podatkov

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import glob
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report
import joblib

testni_podatki = pd.read_csv(
    "podatki/ovrednoteni_podatki/m0.csv", 
    sep=",",           
    decimal=".",       
    header=None,       
    encoding="cp1250"
)

testni_podatki.columns
sirina,visina = testni_podatki.shape
sirina
testni_podatki.head(5)

,0,1,2,3
0,0,01.03 00:00,27.713,1
1,0,01.03 01:00,27.784,1
2,0,01.03 02:00,27.733,1
3,0,01.03 03:00,27.737,1
4,0,01.03 04:00,27.764,1


In [2]:

path = 'podatki/ovrednoteni_podatki/'
all_files = glob.glob(path + "m*.csv") 


li = []
for filename in all_files:
    df_temp = pd.read_csv(filename, sep=",", decimal=".", header=None, encoding="cp1250")
   
    li.append(df_temp)

df_oznaceno = pd.concat(li, axis=0, ignore_index=True)

df_oznaceno.columns = ["ID Merilnika", "Čas (DD.MM HH)", "meritev", "anomalija"]

df_oznaceno.head(-5)

,ID Merilnika,Čas (DD.MM HH),meritev,anomalija
0,84,01.03 00:00,12.916,0
1,84,01.03 01:00,14.424,0
2,84,01.03 02:00,14.321,0
3,84,01.03 03:00,12.968,0
4,84,01.03 04:00,11.593,0
...,...,...,...,...
47909,59,10.03 15:00,20.282,0
47910,59,10.03 16:00,20.516,0
47911,59,10.03 17:00,22.363,0
47912,59,10.03 18:00,30.292,0


In [3]:
df_oznaceno["anomalija"].value_counts()

anomalija
0    43996
1     3923
Name: count, dtype: int64

## Dodatni parametri za izboljšanje rezultata

In [4]:
df_oznaceno = df_oznaceno.sort_values(['ID Merilnika', 'Čas (DD.MM HH)'])

for i in range(1, 4):
    df_oznaceno[f'lag_{i}'] = df_oznaceno.groupby('ID Merilnika')['meritev'].shift(i)

df_oznaceno['rolling_mean'] = df_oznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).mean())
df_oznaceno['rolling_std'] = df_oznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).std())

df_oznaceno['velocity'] = df_oznaceno.groupby('ID Merilnika')['meritev'].diff()


In [5]:
df_clean = df_oznaceno.dropna().copy()

features = ['meritev', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean', 'rolling_std', 'velocity']

df_clean = df_clean.sort_values('Čas (DD.MM HH)')

## Učenje in vrednotenje modela

In [6]:
df_clean = df_clean.sort_values('Čas (DD.MM HH)')

X = df_clean[features]
y = df_clean['anomalija']

tscv = TimeSeriesSplit(n_splits=5)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    model = xgb.XGBClassifier(
 
        random_state=42)
    model.fit(X_train, y_train)
    
    probs = model.predict_proba(X_test)[:, 1]
    predictions = (probs > 0.35).astype(int)
    print(f"Fold Results:\n{classification_report(y_test, predictions)}")

Fold Results:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7324
           1       0.90      0.92      0.91       529

    accuracy                           0.99      7853
   macro avg       0.95      0.96      0.95      7853
weighted avg       0.99      0.99      0.99      7853

Fold Results:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7306
           1       0.92      0.88      0.90       547

    accuracy                           0.99      7853
   macro avg       0.96      0.94      0.95      7853
weighted avg       0.99      0.99      0.99      7853

Fold Results:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      7392
           1       0.85      0.96      0.90       461

    accuracy                           0.99      7853
   macro avg       0.93      0.98      0.95      7853
weighted avg       0.99      0.9

## Pregled motenj in njihovih dolžin

In [7]:
probs = model.predict_proba(X)[:, 1]
df_clean['pred_anom'] = (probs > 0.35).astype(int)

df_clean['transition'] = df_clean.groupby('ID Merilnika')['pred_anom'].diff()

interruption_list = []

for house_id, group in df_clean.groupby('ID Merilnika'):
    starts = group[group['transition'] == 1]['Čas (DD.MM HH)'].tolist()
    ends = group[group['transition'] == -1]['Čas (DD.MM HH)'].tolist()
  
    for i in range(min(len(starts), len(ends))):
        interruption_list.append({
            'Meter_ID': house_id,
            'Start_Time': starts[i],
            'End_Time': ends[i]
        })

df_report = pd.DataFrame(interruption_list)

df_report['Start_Time'] = pd.to_datetime(
    df_report['Start_Time'].str.replace(' ', '.2026 '), 
    format='%d.%m.%Y %H:%M'
)
df_report['End_Time'] = pd.to_datetime(
    df_report['End_Time'].str.replace(' ', '.2026 '), 
    format='%d.%m.%Y %H:%M'
)

df_report['Duration'] = df_report['End_Time'] - df_report['Start_Time']

df_report = df_report.sort_values(by='Duration', ascending=False)
print(df_report.head(10))

    Meter_ID          Start_Time            End_Time        Duration
6         11 2026-03-08 13:00:00 2026-03-10 23:00:00 2 days 10:00:00
21        18 2026-03-08 15:00:00 2026-03-10 16:00:00 2 days 01:00:00
30        23 2026-03-08 14:00:00 2026-03-10 05:00:00 1 days 15:00:00
18        16 2026-03-08 13:00:00 2026-03-10 01:00:00 1 days 12:00:00
7         12 2026-03-08 13:00:00 2026-03-09 14:00:00 1 days 01:00:00
83        79 2026-03-05 01:00:00 2026-03-06 00:00:00 0 days 23:00:00
23        21 2026-03-08 14:00:00 2026-03-09 11:00:00 0 days 21:00:00
84        83 2026-03-06 04:00:00 2026-03-07 00:00:00 0 days 20:00:00
85        83 2026-03-10 04:00:00 2026-03-11 00:00:00 0 days 20:00:00
19        16 2026-03-10 02:00:00 2026-03-10 20:00:00 0 days 18:00:00


In [8]:

path = 'podatki/vsi_podatki/' 
all_files = glob.glob(path + "m*.csv") 

# 2. Use a list comprehension to read all files at once
# We include the same settings we found for your 'm0.csv'
li = []
for filename in all_files:
    df_temp = pd.read_csv(filename, sep=",", decimal=".", header=None, encoding="cp1250")
    # Optional: Add a column to keep track of which file the data came from
    #df_temp['source_file'] = filename 
    li.append(df_temp)

# 3. Concatenate everything into one big DataFrame
df_neoznaceno = pd.concat(li, axis=0, ignore_index=True)

# 4. Name the columns (using the names we discussed)
df_neoznaceno.columns = ["ID Merilnika", "Čas (DD.MM HH)", "meritev"]

df_neoznaceno.head(-5)

,ID Merilnika,Čas (DD.MM HH),meritev
0,84,01.03 00:00,12.916
1,84,01.03 01:00,14.424
2,84,01.03 02:00,14.321
3,84,01.03 03:00,12.968
4,84,01.03 04:00,11.593
...,...,...,...
1023102,59,30.09 14:00,28.107
1023103,59,30.09 15:00,27.034
1023104,59,30.09 16:00,26.413
1023105,59,30.09 17:00,26.701


## Sortiranje podatkov


In [9]:
df_neoznaceno = df_neoznaceno.sort_values(['ID Merilnika', 'Čas (DD.MM HH)'])

for i in range(1, 4):
    df_neoznaceno[f'lag_{i}'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].shift(i)

df_neoznaceno['rolling_mean'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).mean())
df_neoznaceno['rolling_std'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].transform(lambda x: x.rolling(window=5).std())
df_neoznaceno['velocity'] = df_neoznaceno.groupby('ID Merilnika')['meritev'].diff()

df_predict_clean = df_neoznaceno.dropna().copy()

X_new = df_predict_clean[features]

## Dodajanje mere za izločanje mejnih vrednosti

In [10]:

probs_new = model.predict_proba(X_new)[:, 1]
df_predict_clean['pred_anom'] = (probs_new > 0.35).astype(int)

print(f"Total rows in new data: {len(df_predict_clean)}")
print(f"Anomalous points detected: {df_predict_clean['pred_anom'].sum()}")

Total rows in new data: 1022312
Anomalous points detected: 75602


In [11]:

df_predict_clean['dt_obj'] = pd.to_datetime(
    df_predict_clean['Čas (DD.MM HH)'] + '.2026', 
    format='%d.%m %H:%M.%Y', 
    dayfirst=True
)

df_predict_clean = df_predict_clean.sort_values(['ID Merilnika', 'dt_obj'])

df_predict_clean['transition'] = df_predict_clean.groupby('ID Merilnika')['pred_anom'].diff()

final_anomalies = []
for house_id, group in df_predict_clean.groupby('ID Merilnika'):
    starts = group[group['transition'] == 1]['dt_obj'].tolist()
    ends = group[group['transition'] == -1]['dt_obj'].tolist()
    
    for s, e in zip(starts, ends):
        if e > s: 
            final_anomalies.append({
                'Meter_ID': house_id,
                'Start_Time': s.strftime('%d.%m %H:%M'),
                'End_Time': e.strftime('%d.%m %H:%M'),
                'Duration': e - s
            })

df_submission = pd.DataFrame(final_anomalies)
display(df_submission.sort_values('Duration', ascending=False).head(10))

,Meter_ID,Start_Time,End_Time,Duration
4375,136,03.08 18:00,12.09 13:00,39 days 19:00:00
908,23,15.03 04:00,21.04 01:00,36 days 21:00:00
469,16,15.03 04:00,21.04 01:00,36 days 21:00:00
333,14,17.03 04:00,21.04 01:00,34 days 21:00:00
13,8,17.03 12:00,21.04 01:00,34 days 13:00:00
810,21,18.03 02:00,21.04 01:00,33 days 23:00:00
116,9,18.03 04:00,21.04 01:00,33 days 21:00:00
531,17,18.03 04:00,21.04 01:00,33 days 21:00:00
182,10,18.03 04:00,21.04 01:00,33 days 21:00:00
278,12,18.03 17:00,21.04 01:00,33 days 08:00:00


## Še en csv za pregled

In [12]:
df_submission[['Meter_ID', 'Start_Time', 'End_Time', 'Duration']].to_csv('final_predictions.csv', index=False)
print("csv narjen")

Results saved to final_predictions.csv


## Izvoz modela za uporabo v spletni aplikaciji

In [14]:
probs_new = model.predict_proba(X_new)[:, 1]
df_predict_clean['pred_anom_raw'] = (probs_new > 0.35).astype(int)

df_predict_clean['pred_anom'] = df_predict_clean.groupby('ID Merilnika')['pred_anom_raw'].transform(
    lambda x: x.rolling(window=3, center=True).median().fillna(0).astype(int)
)

df_predict_clean['transition'] = df_predict_clean.groupby('ID Merilnika')['pred_anom'].diff()

In [15]:
final_events = []
grace_period = pd.Timedelta(hours=2) 

for house_id, group in df_predict_clean.groupby('ID Merilnika'):
    starts = group[group['transition'] == 1]['dt_obj'].tolist()
    ends = group[group['transition'] == -1]['dt_obj'].tolist()
    
    temp_pairs = [{'s': s, 'e': e} for s, e in zip(starts, ends) if e > s]
    if not temp_pairs: continue
    
    merged = []
    curr = temp_pairs[0]
    for i in range(1, len(temp_pairs)):
        nxt = temp_pairs[i]
        if (nxt['s'] - curr['e']) <= grace_period:
            curr['e'] = max(curr['e'], nxt['e'])
        else:
            merged.append(curr)
            curr = nxt
    merged.append(curr)
    
    for event in merged:
        final_events.append({
            'Meter_ID': house_id,
            'Start_Time': event['s'].strftime('%d.%m %H:%M'),
            'End_Time': event['e'].strftime('%d.%m %H:%M'),
            'Duration': event['e'] - event['s'],
            'Duration_hrs': (event['e'] - event['s']).total_seconds() / 3600
        })

df_final_log = pd.DataFrame(final_events)

## Izračun SAIFI in SAIDI

In [16]:
total_customers = df_neoznaceno['ID Merilnika'].nunique()
total_events = len(df_final_log)
total_dur_hrs = df_final_log['Duration_hrs'].sum()

saifi_final = total_events / total_customers
saidi_final_hrs = total_dur_hrs / total_customers

print("--- IMPROVED RELIABILITY REPORT ---")
print(f"Total Users: {total_customers}")
print(f"Total Events: {total_events}")
print("-" * 35)
print(f"SAIFI (Frequency): {saifi_final:.4f}")
print(f"SAIDI (Duration):  {saidi_final_hrs:.2f} hours")

--- IMPROVED RELIABILITY REPORT ---
Total Users: 200
Total Events: 3993
-----------------------------------
SAIFI (Frequency): 19.9650
SAIDI (Duration):  367.48 hours


In [17]:
df_final_log[['Meter_ID', 'Start_Time', 'End_Time', 'Duration']].to_csv('final_predictions_cleaned.csv', index=False)

joblib.dump(model, 'final_xgboost_model.pkl')
joblib.dump(features, 'model_features.pkl')
print("konec z eksportanjem")

All files exported and ready for the web app!


## Izvoz zadnjega csv-ja


In [20]:

df_final = df_predict_clean[['ID Merilnika', 'Čas (DD.MM HH)', 'pred_anom', 'meritev']].copy()

df_final.loc[df_final['meritev'] < 2.0, 'pred_anom'] = 0

df_final['dt_obj'] = df_predict_clean['dt_obj']
df_final = df_final.sort_values(by=['ID Merilnika', 'dt_obj'])

output_data = df_final[['ID Merilnika', 'Čas (DD.MM HH)', 'pred_anom']]

output_data.to_csv('rezultati.csv', header=False, index=False)

print("--- STATUS ODDAJE ---")
print(f"Datoteka 'rezultati.csv' je bila uspešno ustvarjena.")
print(f"Število vrstic: {len(output_data)}")
print(f"Zadnje tri vrstice v datoteki (primer):\n{output_data.tail(3).to_string(header=False, index=False)}")

--- STATUS ODDAJE ---
Datoteka 'rezultati.csv' je bila uspešno ustvarjena.
Število vrstic: 1022312
Zadnje tri vrstice v datoteki (primer):
199 30.09 21:00 0
199 30.09 22:00 0
199 30.09 23:00 0
